# Radial Intensity Profiling of Micropatterned Colonies

**v1.0 — dual-segmenter, dual-mode pipeline.**

For each multichannel TIFF of a micropatterned colony this notebook:

1. Max-projects the Z-stack (or accepts an already-projected `(C, Y, X)` TIFF)
2. Gaussian-smooths + contrast-enhances each channel (`tapenade.global_contrast_enhancement`)
3. Segments nuclei on the DAPI channel — **StarDist** (default, fast on CPU) or **Cellpose-SAM** (better on dense/irregular nuclei; needs a GPU to be practical)
4. Finds the colony center as the center of mass of the nuclear mask
5. Computes radial intensity profiles in **two complementary modes**:
   - **Per-pixel** (classic): mean masked intensity per 1-px radial bin
   - **Per-nucleus** (recommended): one data point per nucleus (distance from center, mean intensity inside that nucleus) + LOWESS trend — this removes the noisy jump near r = 0 caused by tiny bins, and it is the biologically right readout for nuclear factors like SMAD2
6. Saves per-image plots + CSVs, cross-colony summary figures (including a publication-style multi-panel figure), and a PowerPoint report

**Channel assumption (edit in the parameters cell if different):**

| Index | Marker |
|---|---|
| C0 | ZO-1 |
| C1 | DAPI |
| C2 | SMAD2 |


## Imports

In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import tifffile
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.ndimage import gaussian_filter, center_of_mass
from skimage.measure import find_contours, regionprops
from skimage.filters import threshold_otsu
import itertools
from tapenade.preprocessing import global_contrast_enhancement
from statsmodels.nonparametric.smoothers_lowess import lowess
from pptx import Presentation
from pptx.util import Inches


## Parameters — everything tunable lives here

In [ ]:
# === IMAGING (change for your microscope) ===
SCALE_UM_PER_PX = 0.5
CHANNEL_NAMES   = ['ZO-1', 'DAPI', 'SMAD2']
DAPI_IDX        = 1

# === SEGMENTATION BACKEND ===
# 'stardist' — fast on CPU (~2-3 s/image), great for typical hESC colonies
# 'cellpose' — Cellpose-SAM, more robust on dense/irregular nuclei, but slow
#              without a GPU (minutes per image on CPU)
SEGMENTER = 'stardist'

# --- StarDist settings ---
STARDIST_NORM_PERCENTILES = (1, 99.8)

# --- Cellpose settings (used only when SEGMENTER='cellpose') ---
# Missing nuclei?  lower CELLPROB (-1) / FLOW (0.3).  Extra junk? raise CELLPROB (+1) / MIN_SIZE.
CELLPOSE_DIAMETER_PX        = 35
CELLPOSE_FLOW_THRESHOLD     = 0.4
CELLPOSE_CELLPROB_THRESHOLD = 0.0
CELLPOSE_MIN_SIZE           = 15

# === PREPROCESSING ===
GAUSSIAN_SIGMA       = 1.0
CONTRAST_PERCENTILES = (0.5, 99.5)   # tapenade global contrast enhancement

# === ANALYSIS ===
MAX_RADIUS_UM      = 250    # truncation radius for the per-pixel profile
PLOT_MAX_UM        = 300    # x-axis extent
NORMALIZE_PROFILES = False  # False = keep absolute (a.u.) amplitudes so colonies can be
                            # compared to each other (recommended). True = each colony max -> 1.
BIN_WIDTH_UM       = 10     # radial bin width for per-nucleus summaries
LOWESS_FRAC        = 0.2    # LOWESS smoothing span (fraction of points per fit)
RATIO_TO_DAPI      = True   # also compute Cn/DAPI per nucleus (controls for density/section
                            # thickness; standard for nuclear factors like SMAD2)
BOOTSTRAP_N        = 1000   # colony-level bootstrap resamples for the CI band (>=4 colonies)

# === TAPENADE-PAPER EXTENSIONS (Gros et al., eLife 2026, doi:10.7554/eLife.107154) ===
DAPI_FIELD_NORM     = False  # re-normalize all channels by a masked-Gaussian DAPI field
                             # (2D version of their Fig. 5 optical-artifact correction)
FIELD_NORM_SIGMA_UM = 12     # field kernel ~ nucleus diameter (paper: 10-15 um optimal)
POSITIVE_FRACTION   = True   # Otsu per colony -> fraction of positive nuclei vs radius
                             # (their approach for sparse markers like FoxA2)
COEXPRESSION_PLOTS  = True   # per-nucleus pairwise co-expression histograms (their Fig. 5f)

# === PLOT STYLE ===
CHANNEL_COLORS = {'ZO-1': '#00A087', 'DAPI': '#3C5488', 'SMAD2': '#E64B35'}  # npg palette
CHANNEL_CMAPS  = ['plasma', 'inferno', 'magma']

NATURE_RC = {
    'font.family': 'DejaVu Sans', 'font.size': 8,
    'axes.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.frameon': False,
    'pdf.fonttype': 42,
}


## Segmentation backend (lazy-loaded)

In [ ]:
_seg_fn = None

def get_segmenter():
    """Load the chosen segmentation model once; return a function img -> label image."""
    global _seg_fn
    if _seg_fn is not None:
        return _seg_fn

    if SEGMENTER == 'stardist':
        from stardist.models import StarDist2D
        from csbdeep.utils import normalize as _sd_normalize
        model = StarDist2D.from_pretrained('2D_versatile_fluo')

        def _segment(dapi):
            labels, _ = model.predict_instances(
                _sd_normalize(dapi, *STARDIST_NORM_PERCENTILES, clip=True))
            return labels

    elif SEGMENTER == 'cellpose':
        from cellpose import models as _cp_models
        model = _cp_models.CellposeModel(gpu=False, pretrained_model='cpsam')

        def _segment(dapi):
            masks, _flows, _styles = model.eval(
                dapi,
                diameter=CELLPOSE_DIAMETER_PX,
                flow_threshold=CELLPOSE_FLOW_THRESHOLD,
                cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
                min_size=CELLPOSE_MIN_SIZE,
                normalize=True)
            return masks

    else:
        raise ValueError(f"SEGMENTER must be 'stardist' or 'cellpose', got {SEGMENTER!r}")

    _seg_fn = _segment
    return _seg_fn


## Helpers

In [ ]:
def save_figure(fig, path, show=True):
    fig.savefig(path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)


def clean_label(name):
    """Readable label from filename; adjust the regex for your naming convention."""
    base = os.path.basename(name)
    for suffix in ('.tif', '.tiff'):
        base = base.replace(suffix, '')
    match = re.search(r'_(.*?)_488', base)
    return match.group(1).replace('_', ' ') if match else base


## Loading, preprocessing, segmentation

In [ ]:
def load_and_preprocess(path):
    """Accept 4D (Z, C, Y, X) or already-projected 3D (C, Y, X) TIFFs.
    Returns (img_enhanced, img_raw) as (C, Y, X) float32."""
    img = tifffile.imread(path)
    if img.ndim == 4:
        img = img.max(axis=0)
    elif img.ndim != 3:
        raise ValueError(f'Expected 3D (C,Y,X) or 4D (Z,C,Y,X) image, got shape {img.shape}')
    img = img.astype(np.float32)
    if img.shape[0] != len(CHANNEL_NAMES):
        raise ValueError(
            f'Expected {len(CHANNEL_NAMES)} channels {CHANNEL_NAMES}, got {img.shape[0]}')
    img_blur = np.stack([gaussian_filter(img[c], sigma=GAUSSIAN_SIGMA)
                         for c in range(img.shape[0])])
    img_enh = np.stack([
        global_contrast_enhancement(img_blur[c],
                                    perc_low=CONTRAST_PERCENTILES[0],
                                    perc_high=CONTRAST_PERCENTILES[1])
        for c in range(img.shape[0])])
    return img_enh, img


def segment_nuclei(img_enh):
    """Segment nuclei on the DAPI channel. Returns (labels, mask, center_xy_float)."""
    labels = get_segmenter()(img_enh[DAPI_IDX])
    mask = labels > 0
    if not mask.any():
        raise RuntimeError('No nuclei found — check DAPI channel / segmenter settings')
    cy, cx = center_of_mass(mask)
    return labels, mask.astype(np.uint8), (cx, cy)


## Mode 1 — per-pixel radial profile (classic)

Mean masked intensity per 1-px radial bin. The truncation past `MAX_RADIUS_UM` keeps the
original behaviour of the V3 pipeline: beyond the cutoff only values at or below the
regional minimum are kept, the rest become NaN and are interpolated for display.


In [ ]:
def radial_profile_masked(data, center, mask):
    y, x = np.indices(data.shape)
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2).astype(int)
    valid = mask.astype(bool)
    counts = np.bincount(r[valid].ravel())
    sums   = np.bincount(r[valid].ravel(), data[valid].ravel())
    profile = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    radii_um = np.arange(len(profile)) * SCALE_UM_PER_PX
    return radii_um, profile


def truncate_profile(radii_um, profile):
    """Original V3 truncation: past MAX_RADIUS_UM keep only values <= the regional
    minimum; everything else becomes NaN (interpolated later for plotting)."""
    profile = profile.copy()
    post = radii_um > MAX_RADIUS_UM
    if post.any():
        vals = profile[post]
        if np.any(np.isfinite(vals)):
            vals = np.where(vals > np.nanmin(vals), np.nan, vals)
            profile[post] = vals
    return profile


## Mode 2 — per-nucleus profile (recommended)

One data point per nucleus: distance of the nucleus centroid from the colony center vs the
mean enhanced intensity inside that nucleus. Then a LOWESS trend per colony.

Why this is better than per-pixel binning:

- **No start-up jump.** The innermost 1-px rings contain only a handful of pixels, so the
  classic profile is wildly noisy near r = 0. Nuclei are natural averaging units and there
  simply are no nuclei at r ≈ 0, so the curve starts where data exists.
- **Biologically correct for nuclear readouts.** SMAD2 signaling activity *is* nuclear
  intensity per nucleus — this is how the micropattern papers (Etoc 2016, Chhabra 2019)
  quantify it.
- **Statistics for free.** n = nuclei per bin → honest SEM bands.


In [ ]:
def per_nucleus_table(img_enh, labels, center):
    """One row per nucleus: position, distance, area, shape metrics, mean intensity/channel.

    Shape metrics follow Gros et al. (eLife 2026): nuclei as proxies for cell deformation.
    cos2_radial = squared cosine between the nucleus major axis and the radial direction
    (1 = radially aligned, 0 = circumferential, 0.5 = random baseline in 2D)."""
    ids = np.arange(1, int(labels.max()) + 1)
    ones = np.ones(labels.shape, dtype=np.float32)
    coms = ndimage.center_of_mass(ones, labels, ids)
    areas_px = ndimage.sum_labels(ones, labels, ids)
    ys = np.array([p[0] for p in coms]); xs = np.array([p[1] for p in coms])
    dist_um = np.sqrt((xs - center[0])**2 + (ys - center[1])**2) * SCALE_UM_PER_PX

    ecc = np.full(len(ids), np.nan)
    cos2 = np.full(len(ids), np.nan)
    major_um = np.full(len(ids), np.nan)
    for p in regionprops(labels):
        j = p.label - 1
        if j >= len(ids):
            continue
        ecc[j] = p.eccentricity
        major_um[j] = p.axis_major_length * SCALE_UM_PER_PX
        u = np.array([np.sin(p.orientation), np.cos(p.orientation)])   # major axis (x, y)
        v = np.array([xs[j] - center[0], ys[j] - center[1]])
        n = np.linalg.norm(v)
        if n > 0:
            cos2[j] = float((u @ (v / n)) ** 2)

    out = {
        'nucleus_id': ids, 'x_px': xs, 'y_px': ys,
        'distance_um': dist_um,
        'area_um2': areas_px * SCALE_UM_PER_PX**2,
        'eccentricity': ecc, 'major_axis_um': major_um, 'cos2_radial': cos2,
    }
    for ch in range(img_enh.shape[0]):
        out[f'C{ch}'] = ndimage.mean(img_enh[ch], labels, ids)
    df = pd.DataFrame(out)
    q99 = df['distance_um'].quantile(0.99)
    df['rel_distance'] = df['distance_um'] / q99 if q99 else np.nan
    if RATIO_TO_DAPI:
        dapi = df[f'C{DAPI_IDX}'].replace(0, np.nan)
        for ch in range(img_enh.shape[0]):
            if ch != DAPI_IDX:
                df[f'C{ch}_over_DAPI'] = df[f'C{ch}'] / dapi
    return df


## Optional: DAPI-field intensity re-normalization (Gros et al. 2026)

The Tapenade paper corrects optical/illumination artifacts by dividing each channel by a
**masked-Gaussian coarse-grained map of the ubiquitous nuclear stain** (their Fig. 5). The 2D
version here corrects uneven illumination across the colony. After normalization the DAPI
channel should be nearly flat — a built-in QC that the correction worked. Off by default
(`DAPI_FIELD_NORM`) to preserve backward comparability.


In [ ]:
def dapi_field_normalize(img_enh, mask):
    """Divide all channels by a masked-Gaussian DAPI field (2D of Gros et al. Fig. 5)."""
    sigma_px = FIELD_NORM_SIGMA_UM / SCALE_UM_PER_PX
    m = mask.astype(np.float32)
    den = gaussian_filter(m, sigma_px)
    num = gaussian_filter(img_enh[DAPI_IDX] * m, sigma_px)
    field = np.where(den > 1e-3, num / np.maximum(den, 1e-3), np.nan)
    ref = np.nanmean(field[mask.astype(bool)])
    field = np.where(np.isfinite(field) & (field > 0), field, ref)
    return img_enh * (ref / field)[None, :, :]


## Per-image analysis

In [ ]:
def full_analysis(path, out_root, plots_root):
    img_name = os.path.basename(path)
    stem = re.sub(r'\.tiff?$', '', img_name, flags=re.IGNORECASE)
    plot_dir = os.path.join(plots_root, stem)
    os.makedirs(plot_dir, exist_ok=True)

    img_enh, img_raw = load_and_preprocess(path)
    labels, mask, center = segment_nuclei(img_enh)
    if DAPI_FIELD_NORM:
        img_enh = dapi_field_normalize(img_enh, mask)
    n_nuclei = int(labels.max())

    H, W = img_enh[DAPI_IDX].shape
    extent = [-W/2*SCALE_UM_PER_PX, W/2*SCALE_UM_PER_PX,
              -H/2*SCALE_UM_PER_PX, H/2*SCALE_UM_PER_PX]

    # --- Max projections ---
    fig, axs = plt.subplots(1, img_raw.shape[0], figsize=(4*img_raw.shape[0], 4))
    for i in range(img_raw.shape[0]):
        axs[i].imshow(img_raw[i], cmap='gray')
        axs[i].set_title(f'Max Projection: {CHANNEL_NAMES[i]}')
        axs[i].axis('off')
    save_figure(fig, os.path.join(plot_dir, 'max_projection.png'))

    # --- Segmentation overlay ---
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_enh[DAPI_IDX], cmap='gray', extent=extent, origin='lower')
    for cnt in find_contours(labels, 0.5):
        ax.plot((cnt[:, 1] - W/2)*SCALE_UM_PER_PX,
                (cnt[:, 0] - H/2)*SCALE_UM_PER_PX, lw=1, color='lime')
    ax.plot((center[0] - W/2)*SCALE_UM_PER_PX, (center[1] - H/2)*SCALE_UM_PER_PX,
            'y+', markersize=20, mew=3)
    ax.set_title(f'DAPI + {SEGMENTER} ({n_nuclei} nuclei)')
    ax.axis('equal')
    save_figure(fig, os.path.join(plot_dir, 'segmentation.png'))

    # --- Heatmaps ---
    fig, axs = plt.subplots(1, img_enh.shape[0], figsize=(4*img_enh.shape[0], 4))
    for i in range(img_enh.shape[0]):
        axs[i].imshow(img_enh[i], cmap=CHANNEL_CMAPS[i], extent=extent, origin='lower')
        axs[i].set_title(CHANNEL_NAMES[i])
        axs[i].axis('equal')
    plt.tight_layout()
    save_figure(fig, os.path.join(plot_dir, 'heatmaps.png'))

    # --- Mode 1: per-pixel profiles ---
    ylabel = 'Normalized Intensity' if NORMALIZE_PROFILES else 'Intensity (a.u.)'
    pix_profiles, full_radius = {}, None
    for c in range(img_enh.shape[0]):
        radii, prof = radial_profile_masked(img_enh[c], center, mask)
        if NORMALIZE_PROFILES:
            peak = np.nanmax(prof)
            if peak and peak > 0:
                prof = prof / peak
        prof = truncate_profile(radii, prof)
        if NORMALIZE_PROFILES:
            peak = np.nanmax(prof)
            if peak and peak > 0:
                prof = prof / peak
        pix_profiles[f'C{c}'] = prof
        if full_radius is None:
            full_radius = radii

        name = CHANNEL_NAMES[c]
        fig, ax = plt.subplots()
        ax.plot(radii, pd.Series(prof).interpolate(limit_direction='both'),
                lw=2, color=CHANNEL_COLORS[name])
        ax.set_xlim(0, PLOT_MAX_UM)
        ax.set_xlabel('Distance from Center (µm)')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{name} Radial Profile (per-pixel)')
        save_figure(fig, os.path.join(plot_dir, f'profile_C{c}.png'))

    pix_df = pd.DataFrame(pix_profiles)
    pix_df.insert(0, 'Radius', full_radius)
    pix_df.to_csv(os.path.join(out_root, f'{stem}_radial.csv'), index=False)

    fig, ax = plt.subplots(figsize=(6, 4))
    for c in range(img_enh.shape[0]):
        name = CHANNEL_NAMES[c]
        ax.plot(full_radius,
                pd.Series(pix_profiles[f'C{c}']).interpolate(limit_direction='both'),
                lw=2, label=name, color=CHANNEL_COLORS[name])
    ax.set_xlim(0, PLOT_MAX_UM)
    ax.set_xlabel('Distance from Center (µm)')
    ax.set_ylabel(ylabel)
    ax.set_title('Combined Radial Profiles (per-pixel)')
    ax.legend()
    save_figure(fig, os.path.join(plot_dir, 'combined_profiles.png'))

    # --- Mode 2: per-nucleus + LOWESS ---
    nuc_df = per_nucleus_table(img_enh, labels, center)
    nuc_df.to_csv(os.path.join(out_root, f'{stem}_nuclei.csv'), index=False)

    fig, axs = plt.subplots(1, img_enh.shape[0], figsize=(5*img_enh.shape[0], 4))
    for c in range(img_enh.shape[0]):
        name = CHANNEL_NAMES[c]
        axs[c].scatter(nuc_df['distance_um'], nuc_df[f'C{c}'],
                       s=4, alpha=0.3, color='gray')
        if len(nuc_df) >= 30:
            sm = lowess(nuc_df[f'C{c}'].values, nuc_df['distance_um'].values,
                        frac=LOWESS_FRAC, return_sorted=True)
            axs[c].plot(sm[:, 0], sm[:, 1], color=CHANNEL_COLORS[name], lw=2.5)
        axs[c].set_xlim(0, PLOT_MAX_UM)
        axs[c].set_xlabel('Distance from Center (µm)')
        axs[c].set_ylabel(f'{name} (a.u.)')
        axs[c].set_title(f'{name} per-nucleus ({len(nuc_df)} nuclei)')
    plt.tight_layout()
    save_figure(fig, os.path.join(plot_dir, 'per_nucleus_profiles.png'))

    return pix_df.assign(Image=img_name), nuc_df.assign(Image=img_name), plot_dir


## Quantitative feature extraction

Curves are for looking at; statistics need numbers. For each colony this extracts:

- **peak_radius_um / peak_value** — where and how high the LOWESS-smoothed profile peaks
- **halfmax_radius_um** — radius where the profile first crosses 50% of its dynamic range
  (a robust "boundary position" for edge-restricted signals like SMAD2)
- **center_edge_ratio** — mean signal in the inner 25% of the colony radius vs the outer 25%
  (single number that captures "edge-high" vs "center-high" patterning)
- **angular_cv** — coefficient of variation of the signal across 12 angular sectors
  (QC: high values flag off-center or asymmetric colonies whose radial average is suspect)


In [ ]:
def extract_features(nuc_df, channel_col, colony_radius_um=None):
    """Per-colony summary features from the per-nucleus table for one channel."""
    d = nuc_df.dropna(subset=[channel_col]).sort_values('distance_um')
    if len(d) < 30:
        return None
    r_max = colony_radius_um or d['distance_um'].quantile(0.99)
    sm = lowess(d[channel_col].values, d['distance_um'].values,
                frac=LOWESS_FRAC, return_sorted=True)
    r_s, v_s = sm[:, 0], sm[:, 1]
    in_range = r_s <= r_max
    r_s, v_s = r_s[in_range], v_s[in_range]
    if len(v_s) < 10:
        return None

    i_pk = int(np.nanargmax(v_s))
    v_lo, v_hi = np.nanmin(v_s), np.nanmax(v_s)
    half = v_lo + 0.5 * (v_hi - v_lo)
    above = np.where(v_s >= half)[0]
    halfmax_r = float(r_s[above[0]]) if len(above) else np.nan

    inner = d[d['distance_um'] <= 0.25 * r_max][channel_col].mean()
    outer = d[d['distance_um'] >= 0.75 * r_max][channel_col].mean()

    return {
        'n_nuclei': len(d),
        'colony_radius_um': float(r_max),
        'peak_radius_um': float(r_s[i_pk]),
        'peak_value': float(v_s[i_pk]),
        'halfmax_radius_um': halfmax_r,
        'center_edge_ratio': float(inner / outer) if outer else np.nan,
    }


def angular_cv(nuc_df, channel_col, center_xy=None, n_sectors=12):
    """QC: coefficient of variation of per-sector mean signal. High = asymmetric colony.
    Uses nucleus positions relative to the distance-weighted centroid of nuclei."""
    d = nuc_df.dropna(subset=[channel_col])
    if len(d) < n_sectors * 3 or 'x_px' not in d.columns:
        return np.nan
    ang = np.arctan2(d['y_px'] - d['y_px'].mean(), d['x_px'] - d['x_px'].mean())
    sector = ((ang + np.pi) / (2 * np.pi) * n_sectors).astype(int).clip(0, n_sectors - 1)
    means = d.groupby(sector)[channel_col].mean()
    return float(means.std() / means.mean()) if means.mean() else np.nan


## PowerPoint summary slide

In [ ]:
def add_image_summary_slide(prs, plot_dir, image_name):
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    tb = slide.shapes.add_textbox(Inches(0.3), Inches(0.05), Inches(12), Inches(0.5))
    tb.text_frame.text = f'Results for {image_name}'
    layout = {
        'combined_profiles.png':   (8.0, 0.5),
        'profile_C0.png':          (0.3, 1.2),
        'profile_C1.png':          (3.0, 1.2),
        'profile_C2.png':          (5.7, 1.2),
        'max_projection.png':      (0.3, 3.9),
        'segmentation.png':        (3.0, 3.9),
        'per_nucleus_profiles.png':(5.7, 3.9),
    }
    for fname, (x, y) in layout.items():
        p = os.path.join(plot_dir, fname)
        if os.path.exists(p):
            slide.shapes.add_picture(p, Inches(x), Inches(y),
                                     width=Inches(2.5), height=Inches(2.5))


## Batch run

Point `BASE_DIR` at your folder of `.tif` files and run. Outputs land under `BASE_DIR`:

```
radial_outputs/   <image>_radial.csv (per-pixel) + <image>_nuclei.csv (per-nucleus)
plots/<image>/    per-image PNGs
summary/          combined CSVs, cross-colony figures, publication figure (PNG+PDF),
                  failed_images.log
ppt_reports/      batch_report.pptx
```


In [ ]:
# CHANGE THIS to point at your folder of TIFF images
BASE_DIR = 'PATH_TO_YOUR_TIFF_FOLDER'

if BASE_DIR == 'PATH_TO_YOUR_TIFF_FOLDER':
    print('⚠ Set BASE_DIR above to your TIFF folder path, then re-run this cell.')
else:
    out_root     = os.path.join(BASE_DIR, 'radial_outputs')
    plots_root   = os.path.join(BASE_DIR, 'plots')
    ppt_root     = os.path.join(BASE_DIR, 'ppt_reports')
    summary_root = os.path.join(BASE_DIR, 'summary')
    for d in (out_root, plots_root, ppt_root, summary_root):
        os.makedirs(d, exist_ok=True)

    tifs = sorted(glob.glob(os.path.join(BASE_DIR, '*.tif')) +
                  glob.glob(os.path.join(BASE_DIR, '*.tiff')))
    print(f'Found {len(tifs)} TIFF file(s); segmenter = {SEGMENTER}')

    failure_log = os.path.join(summary_root, 'failed_images.log')
    open(failure_log, 'w').close()

    pix_all, nuc_all = [], []
    batch_ppt = Presentation()

    for fpath in tifs:
        fname = os.path.basename(fpath)
        try:
            pix_df, nuc_df, plot_dir = full_analysis(fpath, out_root, plots_root)
            pix_all.append(pix_df)
            nuc_all.append(nuc_df)
            add_image_summary_slide(batch_ppt, plot_dir, fname)
            print(f'  ✓ {fname} ({nuc_df.shape[0]} nuclei)')
        except Exception as e:
            with open(failure_log, 'a') as f:
                f.write(f'{fname}\t{type(e).__name__}: {e}\n')
            print(f'  ✗ {fname} — {type(e).__name__}: {e}  (logged)')

    if pix_all:
        pix = pd.concat(pix_all, ignore_index=True)
        nuc = pd.concat(nuc_all, ignore_index=True)
        pix.to_csv(os.path.join(summary_root, 'combined_radial_profiles.csv'), index=False)
        nuc.to_csv(os.path.join(summary_root, 'combined_per_nucleus.csv'), index=False)

        ylabel = 'Normalized Intensity' if NORMALIZE_PROFILES else 'Intensity (a.u.)'

        # Cross-colony per-channel figures (per-pixel, classic style)
        for c, name in enumerate(CHANNEL_NAMES):
            fig, ax = plt.subplots(figsize=(8, 6))
            for img_name, grp in pix.groupby('Image'):
                ax.plot(grp['Radius'],
                        pd.Series(grp[f'C{c}'].values).interpolate(limit_direction='both'),
                        lw=1.5, alpha=0.8, label=clean_label(img_name))
            ax.set_title(f'Combined Radial Profiles - {name}')
            ax.set_xlabel('Distance from Center (µm)')
            ax.set_ylabel(f'{name} {ylabel}')
            ax.set_xlim(0, PLOT_MAX_UM)
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend(fontsize=7, loc='lower right')
            fig.savefig(os.path.join(summary_root, f'combined_profiles_{name}.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Per-colony quantitative features + QC
        feat_rows = []
        for img_name, grp in nuc.groupby('Image'):
            for c, name in enumerate(CHANNEL_NAMES):
                feats = extract_features(grp, f'C{c}')
                if feats is None:
                    continue
                feats.update({'Image': img_name, 'Channel': name,
                              'angular_cv': angular_cv(grp, f'C{c}')})
                feat_rows.append(feats)
        feat_df = pd.DataFrame(feat_rows)
        feat_df.to_csv(os.path.join(summary_root, 'colony_features.csv'), index=False)
        print('\nPer-colony features (mean ± SD across colonies):')
        for name in CHANNEL_NAMES:
            sub = feat_df[feat_df['Channel'] == name]
            if len(sub):
                print(f"  {name}: peak at {sub['peak_radius_um'].mean():.0f}±"
                      f"{sub['peak_radius_um'].std():.0f} µm, half-max boundary "
                      f"{sub['halfmax_radius_um'].mean():.0f}±{sub['halfmax_radius_um'].std():.0f} µm, "
                      f"center:edge {sub['center_edge_ratio'].mean():.2f}")
        high_cv = feat_df[feat_df['angular_cv'] > 0.25]['Image'].unique()
        if len(high_cv):
            print(f'  ⚠ QC: high angular asymmetry (CV>0.25) in: {list(high_cv)}')

        # Publication-style multi-panel figure (per-nucleus + LOWESS)
        nuc_c = nuc[nuc['distance_um'] <= PLOT_MAX_UM].copy()
        nuc_c['bin'] = (nuc_c['distance_um'] // BIN_WIDTH_UM) * BIN_WIDTH_UM + BIN_WIDTH_UM/2
        n_col = nuc_c['Image'].nunique()

        def colony_bootstrap_band(nuc_binned, col, n_boot=BOOTSTRAP_N):
            """95% CI of the binned mean, resampling COLONIES (the honest unit)."""
            piv = nuc_binned.groupby(['Image', 'bin'])[col].mean().unstack('bin')
            if len(piv) < 4:
                return None
            rng = np.random.default_rng(0)
            idx = rng.integers(0, len(piv), size=(n_boot, len(piv)))
            boots = np.nanmean(piv.values[idx], axis=1)
            lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
            return piv.columns.values, np.nanmean(piv.values, axis=0), lo, hi

        with mpl.rc_context(NATURE_RC):
            fig, axes = plt.subplots(1, len(CHANNEL_NAMES), figsize=(7.2, 2.4))
            for ax, (c, name) in zip(np.atleast_1d(axes), enumerate(CHANNEL_NAMES)):
                color = CHANNEL_COLORS[name]
                for img_name, grp in nuc_c.groupby('Image'):
                    grp = grp.sort_values('distance_um')
                    if len(grp) < 30:
                        continue
                    sm = lowess(grp[f'C{c}'].values, grp['distance_um'].values,
                                frac=LOWESS_FRAC, return_sorted=True)
                    ax.plot(sm[:, 0], sm[:, 1], color=color, lw=0.5, alpha=0.25)
                band = colony_bootstrap_band(nuc_c, f'C{c}')
                if band is not None:
                    bx, bmean, blo, bhi = band
                    ax.fill_between(bx, blo, bhi, color=color, alpha=0.25, linewidth=0)
                    ax.plot(bx, bmean, color=color, lw=1.8)
                else:
                    by = nuc_c.groupby('bin')[f'C{c}'].agg(['mean', 'sem']).reset_index()
                    ax.fill_between(by['bin'], by['mean']-by['sem'], by['mean']+by['sem'],
                                    color=color, alpha=0.25, linewidth=0)
                    ax.plot(by['bin'], by['mean'], color=color, lw=1.8)
                ax.set_xlim(0, PLOT_MAX_UM)
                ax.set_xlabel('Distance from center (µm)')
                ax.set_ylabel(f'{name} (a.u.)')
                ax.set_title(name)
                ax.text(0.03, 0.97, f'n = {n_col} colonies\n{len(nuc_c):,} nuclei',
                        transform=ax.transAxes, va='top', ha='left',
                        fontsize=6.5, color='#333333')
            for ax, letter in zip(np.atleast_1d(axes), 'abcdefg'):
                ax.text(-0.20, 1.02, letter, transform=ax.transAxes,
                        fontsize=11, fontweight='bold', va='top', ha='left')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'publication_figure.png'),
                        dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(os.path.join(summary_root, 'publication_figure.pdf'),
                        bbox_inches='tight', facecolor='white')
            plt.close(fig)

        # Fraction of positive nuclei vs radius (Otsu per colony; Gros et al. 2026 sparse-
        # marker approach — their FoxA2 analysis)
        if POSITIVE_FRACTION:
            fig, axesP = plt.subplots(1, len(CHANNEL_NAMES), figsize=(5*len(CHANNEL_NAMES), 4))
            for ci, name in enumerate(CHANNEL_NAMES):
                ax = np.atleast_1d(axesP)[ci]
                agg_rows = []
                for img_name, grp in nuc_c.groupby('Image'):
                    vals = grp[f'C{ci}'].dropna()
                    if len(vals) < 50:
                        continue
                    thr = threshold_otsu(vals.values)
                    g = grp.copy(); g['pos'] = g[f'C{ci}'] >= thr
                    frac = g.groupby('bin')['pos'].mean()
                    ax.plot(frac.index, frac.values, lw=1, alpha=0.35,
                            color=CHANNEL_COLORS[name])
                    agg_rows.append(g[['bin', 'pos']])
                if agg_rows:
                    agg = pd.concat(agg_rows).groupby('bin')['pos'].mean()
                    ax.plot(agg.index, agg.values, lw=2.5, color=CHANNEL_COLORS[name])
                ax.set_xlim(0, PLOT_MAX_UM); ax.set_ylim(0, 1)
                ax.set_xlabel('Distance from Center (µm)')
                ax.set_ylabel(f'Fraction {name}+ nuclei')
                ax.set_title(f'{name}+ fraction (Otsu per colony)')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'positive_fraction_profiles.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Per-nucleus pairwise co-expression histograms (Gros et al. 2026 Fig. 5f)
        if COEXPRESSION_PLOTS:
            pairs = list(itertools.combinations(range(len(CHANNEL_NAMES)), 2))
            fig, axesX = plt.subplots(1, len(pairs), figsize=(5.2*len(pairs), 4.2))
            for ax, (a, b) in zip(np.atleast_1d(axesX), pairs):
                xa, xb = nuc[f'C{a}'].values, nuc[f'C{b}'].values
                ok = np.isfinite(xa) & np.isfinite(xb)
                h = ax.hist2d(xa[ok], xb[ok], bins=80, cmap='inferno',
                              norm=mpl.colors.LogNorm())
                ax.axvline(threshold_otsu(xa[ok]), color='w', lw=0.8, ls='--')
                ax.axhline(threshold_otsu(xb[ok]), color='w', lw=0.8, ls='--')
                r = np.corrcoef(xa[ok], xb[ok])[0, 1]
                ax.set_xlabel(f'{CHANNEL_NAMES[a]} (a.u.)')
                ax.set_ylabel(f'{CHANNEL_NAMES[b]} (a.u.)')
                ax.set_title(f'{CHANNEL_NAMES[a]} vs {CHANNEL_NAMES[b]}  (r = {r:.2f})')
                fig.colorbar(h[3], ax=ax, label='# nuclei')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'coexpression_per_nucleus.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Nuclear morphometrics vs radius (nuclei as deformation proxies)
        fig, axesM = plt.subplots(1, 2, figsize=(11, 4))
        al = nuc_c.dropna(subset=['cos2_radial'])
        if len(al):
            byA = al.groupby('bin')['cos2_radial'].agg(['mean', 'sem']).reset_index()
            axesM[0].fill_between(byA['bin'], byA['mean']-byA['sem'], byA['mean']+byA['sem'],
                                  alpha=0.3, color='#7E6148')
            axesM[0].plot(byA['bin'], byA['mean'], color='#7E6148', lw=2)
        axesM[0].axhline(0.5, color='gray', ls=':', lw=1)
        axesM[0].set_ylim(0, 1); axesM[0].set_xlim(0, PLOT_MAX_UM)
        axesM[0].set_xlabel('Distance from Center (µm)')
        axesM[0].set_ylabel('cos²(major axis, radial)')
        axesM[0].set_title('Nuclear radial alignment (0.5 = random)')
        ecd = nuc_c.dropna(subset=['eccentricity'])
        if len(ecd):
            byE = ecd.groupby('bin')['eccentricity'].agg(['mean', 'sem']).reset_index()
            axesM[1].fill_between(byE['bin'], byE['mean']-byE['sem'], byE['mean']+byE['sem'],
                                  alpha=0.3, color='#4DBBD5')
            axesM[1].plot(byE['bin'], byE['mean'], color='#4DBBD5', lw=2)
        axesM[1].set_xlim(0, PLOT_MAX_UM)
        axesM[1].set_xlabel('Distance from Center (µm)')
        axesM[1].set_ylabel('Eccentricity')
        axesM[1].set_title('Nuclear elongation vs radius')
        plt.tight_layout()
        fig.savefig(os.path.join(summary_root, 'nuclear_morphometrics.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig)

        batch_ppt.save(os.path.join(ppt_root, 'batch_report.pptx'))
        print(f'\nDone. Summary outputs in {summary_root}')
        print('  combined_radial_profiles.csv  (per-pixel)')
        print('  combined_per_nucleus.csv      (per-nucleus)')
        print('  publication_figure.png / .pdf')
    else:
        print('No images processed successfully — see failed_images.log')


## Group comparison — mean profile ± SD per experimental group

Compare genotypes/treatments (e.g. RUES2 vs 72CAG vs HTTKO, ± drug): each colony's profile
is optionally normalized to its own max, then averaged within its group; the band is ±1 SD
(or SEM) across colonies. Groups are assigned by filename substring. One subplot per panel.


In [ ]:
# Map panel title -> {group label: filename substring}. Edit for your experiment, e.g.:
# GROUP_PANELS = {
#     'Activin':      {'RUES2 ACTIVIN': 'RUES2+ActA', '72CAG ACTIVIN': '72CAG_Act',
#                      'HTTKO ACTIVIN': 'HTTKO_Act'},
#     'Activin+BRD9': {'RUES2 ACTIVIN+BRD9': 'RUES2_BRD9', '72CAG ACTIVIN+BRD9': '72CAG_BRD9',
#                      'HTTKO ACTIVIN+BRD9': 'HTTKO_BRD9'},
# }
GROUP_PANELS    = {'All colonies': {'All': ''}}
GROUP_CHANNEL   = 'C2'    # channel to compare (C2 = SMAD2)
GROUP_NORMALIZE = True    # normalize each colony to its own max before averaging
GROUP_BAND      = 'sd'    # 'sd' (classic "± 1 SD by Group") or 'sem'


def plot_group_comparison(pix, panels=None, channel=None, normalize=None, band=None,
                          out_path=None, max_r=None):
    """Mean per-pixel profile ± band per experimental group, one subplot per panel.
    `pix`: combined per-pixel dataframe (columns: Radius, C*, Image)."""
    panels = panels or GROUP_PANELS
    channel = channel or GROUP_CHANNEL
    normalize = GROUP_NORMALIZE if normalize is None else normalize
    band = band or GROUP_BAND
    max_r = max_r or MAX_RADIUS_UM
    palette = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4']

    with mpl.rc_context(NATURE_RC):
        fig, axes = plt.subplots(1, len(panels), figsize=(4.6*len(panels), 3.4),
                                 squeeze=False)
        for ax, (panel_name, groups) in zip(axes[0], panels.items()):
            for gi, (label, pattern) in enumerate(groups.items()):
                imgs = [im for im in pix['Image'].unique() if pattern in im]
                if not imgs:
                    print(f'  (no images match {label!r} pattern {pattern!r})')
                    continue
                curves = []
                for im in imgs:
                    g = pix[pix['Image'] == im].sort_values('Radius')
                    g = g[g['Radius'] <= max_r]
                    y = pd.Series(g[channel].values).interpolate(limit_direction='both')
                    if normalize and y.max() > 0:
                        y = y / y.max()
                    curves.append(pd.Series(y.values, index=g['Radius'].round(1)))
                mat = pd.concat(curves, axis=1)
                mean = mat.mean(axis=1)
                spread = mat.std(axis=1) if band == 'sd' else mat.sem(axis=1)
                color = palette[gi % len(palette)]
                ax.plot(mean.index, mean.values, color=color, lw=1.8,
                        label=f'{label} (n={len(imgs)})')
                ax.fill_between(mean.index, mean - spread, mean + spread,
                                color=color, alpha=0.25, linewidth=0)
            ax.set_xlabel('Distance (µm)')
            ax.set_ylabel('Normalized Intensity' if normalize else 'Intensity (a.u.)')
            ax.set_title(panel_name)
            ax.legend(title='Group (n)', fontsize=6.5, title_fontsize=6.5,
                      loc='lower left')
            ax.set_xlim(0, max_r + 10)
        plt.tight_layout()
        if out_path:
            fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(out_path.replace('.png', '.pdf'), bbox_inches='tight',
                        facecolor='white')
        plt.show()
    return fig


# Example (after a batch run):
# pix = pd.read_csv(os.path.join(BASE_DIR, 'summary', 'combined_radial_profiles.csv'))
# plot_group_comparison(pix, out_path=os.path.join(BASE_DIR, 'summary',
#                                                  'group_comparison_SMAD2.png'))


---
## Future work: 3D gastruloid segmentation with the pretrained tapenade StarDist3D

The Tapenade group provides their custom **StarDist3D** model (`tapenade_stardist`), trained
on 4,414 annotated gastruloid nuclei (F1 = 85±3%, constant across >200 µm depth). The cells
below download it from Zenodo (12.6 MB, cached locally), load it, and provide a 3D
per-nucleus table that mirrors this pipeline's 2D analysis — with **distance to the sample
border** (the paper's radial coordinate for variable-size 3D samples) instead of distance
to a 2D colony center.

**Model constraints (from their readme):** isotropic voxels, nuclei ≈ 15 px diameter
(so ≈ 1 µm/voxel for typical 10–15 µm nuclei), intensity normalized to [0, 1].
Set `Z_STEP_UM` to your acquisition's z-spacing.

These cells are self-contained and do not run in the 2D batch above.


In [ ]:
TAPENADE_ZENODO_API = 'https://zenodo.org/api/records/14748083'
TAPENADE_MODEL_ZIP  = 'stardist_tapenade_model.zip'   # 12.6 MB (ignore the 9.4 GB data zip)


def load_tapenade_stardist3d(model_root='models'):
    """Download (once, cached) and load the pretrained tapenade StarDist3D model
    (Gros et al., eLife 2026; Zenodo 14748083)."""
    import urllib.request, zipfile
    os.makedirs(model_root, exist_ok=True)
    basedir = os.path.join(model_root, 'stardist_tapenade_model')
    if not os.path.isdir(os.path.join(basedir, 'tapenade_stardist')):
        url = f'{TAPENADE_ZENODO_API}/files/{TAPENADE_MODEL_ZIP}/content'
        zip_path = os.path.join(model_root, TAPENADE_MODEL_ZIP)
        print(f'Downloading {TAPENADE_MODEL_ZIP} from Zenodo…')
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(model_root)
        os.remove(zip_path)
    from stardist.models import StarDist3D
    return StarDist3D(None, name='tapenade_stardist', basedir=basedir)


def segment_nuclei_3d(dapi_zyx, z_step_um, xy_um_per_px=SCALE_UM_PER_PX,
                      target_um_per_vox=1.0, model=None):
    """Segment nuclei in a 3D (Z, Y, X) stack with the tapenade StarDist3D model.

    Rescales to isotropic voxels of `target_um_per_vox` (their model expects nuclei of
    ~15 px diameter, i.e. ~1 um/voxel for 10-15 um nuclei), normalizes, predicts, and
    returns (labels_isotropic, voxel_size_um)."""
    from scipy.ndimage import zoom as _zoom
    if model is None:
        model = load_tapenade_stardist3d()
    factors = (z_step_um / target_um_per_vox,
               xy_um_per_px / target_um_per_vox,
               xy_um_per_px / target_um_per_vox)
    iso = _zoom(dapi_zyx.astype(np.float32), factors, order=1)
    norm = global_contrast_enhancement(iso, perc_low=1, perc_high=99)
    labels, _ = model.predict_instances(norm)
    return labels, target_um_per_vox


In [ ]:
def per_nucleus_table_3d(channels_iso, labels, voxel_um):
    """3D analogue of per_nucleus_table for gastruloids.

    channels_iso : (C, Z, Y, X) intensity channels at the SAME isotropic voxel size
                   as `labels` (rescale them with the same zoom factors).
    labels       : 3D label image from segment_nuclei_3d.
    Returns one row per nucleus: centroid, volume, distance to the sample border
    (the Tapenade paper's radial coordinate), and per-channel mean intensities."""
    from scipy.ndimage import distance_transform_edt, binary_fill_holes, binary_closing

    ids = np.arange(1, int(labels.max()) + 1)
    ones = np.ones(labels.shape, dtype=np.float32)
    coms = ndimage.center_of_mass(ones, labels, ids)
    vol_vox = ndimage.sum_labels(ones, labels, ids)

    # Sample mask: closed + filled union of nuclei; EDT gives distance to border
    sample = binary_fill_holes(binary_closing(labels > 0, np.ones((5, 5, 5))))
    edt = distance_transform_edt(sample) * voxel_um

    zs = np.array([c[0] for c in coms]); ys = np.array([c[1] for c in coms])
    xs = np.array([c[2] for c in coms])
    zi = np.clip(zs.round().astype(int), 0, labels.shape[0]-1)
    yi = np.clip(ys.round().astype(int), 0, labels.shape[1]-1)
    xi = np.clip(xs.round().astype(int), 0, labels.shape[2]-1)

    out = {
        'nucleus_id': ids, 'z_vox': zs, 'y_vox': ys, 'x_vox': xs,
        'volume_um3': vol_vox * voxel_um**3,
        'dist_to_border_um': edt[zi, yi, xi],
    }
    for c in range(channels_iso.shape[0]):
        out[f'C{c}'] = ndimage.mean(channels_iso[c], labels, ids)
    return pd.DataFrame(out)


# --- Example usage on a (Z, C, Y, X) two-photon stack ------------------------------
# from scipy.ndimage import zoom
# Z_STEP_UM = 1.0                                    # <- your z spacing!
# stack = tifffile.imread('my_gastruloid.tif')       # (Z, C, Y, X)
# labels3d, vox = segment_nuclei_3d(stack[:, DAPI_IDX], Z_STEP_UM)
# factors = (Z_STEP_UM / vox, SCALE_UM_PER_PX / vox, SCALE_UM_PER_PX / vox)
# chans = np.stack([zoom(stack[:, c].astype(np.float32), factors, order=1)
#                   for c in range(stack.shape[1])])
# nuc3d = per_nucleus_table_3d(chans, labels3d, vox)
# nuc3d.to_csv('gastruloid_nuclei_3d.csv', index=False)
# # then profile any channel vs nuc3d['dist_to_border_um'] exactly like the 2D LOWESS plots
